# Review links and explore the network

Analysis views use identifier-backed IRS edges, explicit bill references, and human-accepted cross-source name matches.


## Setup


In [1]:
from pathlib import Path
import sqlite3
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
DB_PATH = ROOT / 'data' / 'irs990_full.db'
conn = sqlite3.connect(DB_PATH)

def q(sql, **kw):
    return pd.read_sql_query(sql, conn, **kw)

# Quick row counts across all tables
for tbl in ['irs990_filings','committees','fec_disbursements','lda_filings',
            'lda_lobbying_activities','bills','entity_observations',
            'entity_match_candidates','entity_match_decisions']:
    n = conn.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'{tbl:35} {n:>12,}')


irs990_filings                         4,472,546
committees                                 9,977
fec_disbursements                          2,062
lda_filings                              192,174
lda_lobbying_activities                  697,968
bills                                     16,565
entity_observations                    4,674,697
entity_match_candidates                  185,301
entity_match_decisions                    88,975


## Review name-match candidates

Each row is a candidate pair. Score orders the queue; it is not proof of shared identity. Accept or reject below.


In [2]:
# Distinct org-name pairs ranked by number of matching filing records
# (one org can appear in many filings; GROUP BY collapses to unique org pairs)
n_total = conn.execute('SELECT COUNT(*) FROM entity_match_candidates WHERE is_current=1').fetchone()[0]
n_pairs = conn.execute('''
    SELECT COUNT(*) FROM (
        SELECT lo.normalized_name, ro.normalized_name
        FROM entity_match_candidates c
        JOIN entity_observations lo ON lo.observation_id = c.left_observation_id
        JOIN entity_observations ro ON ro.observation_id = c.right_observation_id
        WHERE c.is_current=1 AND lo.source_system='IRS990'
        GROUP BY lo.normalized_name, ro.normalized_name
    )
''').fetchone()[0]
print(f'Total match candidates (filing-level): {n_total:,}')
print(f'Distinct org pairs:                    {n_pairs:,}')
print()
candidates = q('''
    SELECT MIN(c.candidate_id)         AS candidate_id,
           lo.observed_name            AS irs_name,
           ro.observed_name            AS ext_name,
           ro.source_system            AS ext_source,
           c.matcher_name,
           c.score,
           COUNT(*)                    AS filing_pairs
    FROM entity_match_candidates AS c
    JOIN entity_observations AS lo ON lo.observation_id = c.left_observation_id
    JOIN entity_observations AS ro ON ro.observation_id = c.right_observation_id
    WHERE c.is_current = 1
      AND lo.source_system = \'IRS990\'
    GROUP BY lo.normalized_name, ro.normalized_name, ro.source_system, c.matcher_name
    ORDER BY filing_pairs DESC, c.score DESC
    LIMIT 50
''')
candidates


Total match candidates (filing-level): 85,052
Distinct org pairs:                    2,134



,candidate_id,irs_name,ext_name,ext_source,matcher_name,score,filing_pairs
0,1651,American Legion,AMERICAN LEGION,LDA,normalized_exact_v1,1.0,6188
1,580,UNITED BROTHERHOOD OF CARPENTERS,UNITED BROTHERHOOD OF CARPENTERS,LDA,normalized_exact_v1,1.0,4053
2,2434,INTERNATIONAL BROTHERHOOD OF TEAMSTERS,INTERNATIONAL BROTHERHOOD OF TEAMSTERS,LDA,normalized_exact_v1,1.0,3432
3,505,AMERICAN CIVIL LIBERTIES UNION OF,AMERICAN CIVIL LIBERTIES UNION,LDA,normalized_exact_v1,1.0,3399
4,17,COMMUNICATIONS WORKERS OF AMERICA,COMMUNICATIONS WORKERS OF AMERICA,LDA,normalized_exact_v1,1.0,3056
5,125,AMERICAN FEDERATION OF TEACHERS,AMERICAN FEDERATION OF TEACHERS,LDA,normalized_exact_v1,1.0,3008
6,3157,ASSOCIATED BUILDERS AND CONTRACTORS,ASSOCIATED BUILDERS AND CONTRACTORS INC,LDA,normalized_exact_v1,1.0,2150
7,3053,HABITAT FOR HUMANITY INTERNATIONAL INC,"HABITAT FOR HUMANITY INTERNATIONAL, INC.",LDA,normalized_exact_v1,1.0,1887
8,2099,United Steelworkers,UNITED STEELWORKERS,LDA,normalized_exact_v1,1.0,1576
9,1741,AMERICAN POSTAL WORKERS UNION,AMERICAN POSTAL WORKERS UNION,LDA,normalized_exact_v1,1.0,1407


### How cross-source connections are made

IRS, FEC, and LDA share no common ID. The pipeline connects them two ways:

**1. Name matching (needs human sign-off)**

Every org name from each source is normalized (lowercase, punctuation removed, noise words like 'Inc' and 'LLC' stripped) and stored in `entity_observations`. Pairs with identical normalized names are queued in `entity_match_candidates` with score 1.0. Fuzzy matching is available separately but is slow at this scale.

Nothing reaches the analysis views until a decision is written to `entity_match_decisions`. The `approved_external_entity_links` view assembles `(IRS EIN, source system, external record ID)` triples from accepted decisions only.

**2. Bill references from lobbying text (no review needed)**

LDA activity descriptions often name bills directly, e.g. `HR 2670` or `S.1001`. A regex scans every description and writes matches to `lobbying_bill_links`. No name matching involved.

**What each view requires**

| View | How it joins | Needs accepted matches? |
|---|---|---|
| `grant_network_edges` | IRS Sch. I grantee EINs | No |
| `related_organization_edges` | IRS Sch. R related EINs | No |
| `lobbying_bill_facts` | LDA filing -> bill regex | No |
| `organization_policy_links` | IRS EIN -> accepted LDA filing -> bill | Yes |
| `organization_fec_disbursements` | IRS EIN -> accepted FEC committee -> disbursements | Yes |

**Auto-accepting identical name matches**

The cell below accepts all IRS<->LDA pairs where the org name is the same text, differing only in capitalization (IRS uses title-case; LDA submissions are all-caps). FEC is excluded because generic words like 'Freedom' or 'Fund' produce many false matches between nonprofits and unrelated PACs. Those need the manual review queue above.


In [3]:
# Auto-accept IRS<->LDA candidates where observed names are identical (case-insensitive).
# Idempotent: INSERT OR IGNORE means re-running creates no duplicates.
# FEC candidates excluded -- generic name tokens cause false positives with PACs.

already = conn.execute(
    "SELECT COUNT(*) FROM entity_match_decisions WHERE reviewer='auto-exact-text'"
).fetchone()[0]

if already > 0:
    print(f'Already applied: {already:,} auto-exact-text decisions on record.')
else:
    rows = conn.execute('''
        SELECT c.candidate_id
        FROM entity_match_candidates AS c
        JOIN entity_observations AS lo ON lo.observation_id = c.left_observation_id
        JOIN entity_observations AS ro ON ro.observation_id = c.right_observation_id
        WHERE c.is_current = 1
          AND lo.source_system = 'IRS990'
          AND ro.source_system = 'LDA'
          AND c.score = 1.0
          AND UPPER(lo.observed_name) = UPPER(ro.observed_name)
    ''').fetchall()
    conn.executemany(
        "INSERT OR IGNORE INTO entity_match_decisions "
        "(candidate_id, decision, reviewer, rationale) "
        "VALUES (?, 'accepted', 'auto-exact-text', "
        "'Identical observed_name in IRS990 and LDA (case-insensitive); auto-accepted.')",
        rows
    )
    conn.commit()
    print(f'Accepted {len(rows):,} IRS<->LDA exact-text match candidates.')

n_decisions = conn.execute("SELECT COUNT(*) FROM entity_match_decisions WHERE decision='accepted'").fetchone()[0]
n_links = conn.execute('SELECT COUNT(*) FROM organization_policy_links').fetchone()[0]
print(f'Total accepted decisions:       {n_decisions:,}')
print(f'organization_policy_links rows: {n_links:,}')


Already applied: 88,975 auto-exact-text decisions on record.
Total accepted decisions:       88,975
organization_policy_links rows: 1,180,680


## IRS grant network

Grant flows from Schedule I, joined on grantee EIN. No name matching needed.


In [4]:
grant_edges = q('''
    SELECT source_ein, target_ein, amount, supporting_rows
    FROM grant_network_edges
    ORDER BY amount DESC
    LIMIT 100
''')
print(f'Total grant edges: {conn.execute("SELECT COUNT(*) FROM grant_network_edges").fetchone()[0]:,}')
grant_edges.head(20)


Total grant edges: 2,162,653


,source_ein,target_ein,amount,supporting_rows
0,561776668,560532129,1.406805e+10,8
1,351044585,620646012,9.598521e+09,8
2,383952644,416011702,8.722142e+09,6
3,133971298,135562308,7.948429e+09,7
4,591680273,596002052,4.871932e+09,10
5,271325761,562070036,3.818435e+09,8
6,942829914,943067788,3.203952e+09,8
7,952250801,956006143,2.788579e+09,8
8,946090626,946002123,2.766895e+09,8
9,390743975,396006492,2.718777e+09,42


## IRS related-organization network

Related-org edges from Schedule R, joined on EIN.


In [5]:
related_edges = q('''
    SELECT source_ein, target_ein, supporting_rows
    FROM related_organization_edges
    ORDER BY supporting_rows DESC
    LIMIT 100
''')
print(f'Total related-org edges: {conn.execute("SELECT COUNT(*) FROM related_organization_edges").fetchone()[0]:,}')
related_edges.head(20)


Total related-org edges: 1,331,120


,source_ein,target_ein,supporting_rows
0,530215881,999999999,1109
1,980571483,999999999,616
2,981718091,999999999,616
3,526055213,999999999,543
4,251010829,111111111,496
5,951816009,951816009,464
6,160747359,160747359,424
7,411616861,411616861,408
8,132792409,132792409,259
9,943155150,943155150,242


## LDA lobbying filings

Top spending clients, issue code breakdown, and bill references from activity text.


In [6]:
# Top spenders by total expenses
top_spenders = q('''
    SELECT client_name, SUM(CAST(expenses AS REAL)) AS total_expenses,
           COUNT(*) AS filings
    FROM lda_filings
    WHERE expenses IS NOT NULL
    GROUP BY client_name
    ORDER BY total_expenses DESC
    LIMIT 20
''')
top_spenders


,client_name,total_expenses,filings
0,CHAMBER OF COMMERCE OF THE U.S.A.,160840000.0,9
1,NATIONAL ASSOCIATION OF REALTORS,158650000.0,10
2,AMERICAN HOSPITAL ASSOCIATION,58420000.0,9
3,PHARMACEUTICAL RESEARCH AND MANUFACTURERS OF A...,58090000.0,8
4,AMERICAN MEDICAL ASSOCIATION,50260000.0,9
5,BUSINESS ROUNDTABLE INC,47960000.0,9
6,AARP,44000000.0,10
7,"META PLATFORMS, INC. AND VARIOUS SUBSIDIARIES",43730000.0,8
8,PFIZER INC.,38200000.0,14
9,AMERICAN CHEMISTRY COUNCIL,37890000.0,8


In [7]:
# Issue code distribution
issue_codes = q('''
    SELECT general_issue_code, COUNT(*) AS activity_count
    FROM lda_lobbying_activities
    WHERE general_issue_code IS NOT NULL
    GROUP BY general_issue_code
    ORDER BY activity_count DESC
    LIMIT 30
''')
issue_codes


,general_issue_code,activity_count
0,BUD,84743
1,HCR,54575
2,TAX,46026
3,DEF,39538
4,TRA,29367
5,ENG,27830
6,MMM,25366
7,TRD,23407
8,AGR,20781
9,ENV,20286


## Cross-source views

Populated once matches are accepted above. Run `refresh_analysis_layers(DB_PATH)` in notebook 01 after adding new decisions.


In [8]:
policy_links = q('''
    SELECT ein, client_name, bill_id, title, filing_year,
           reported_lobbying_amount
    FROM organization_policy_links
    ORDER BY reported_lobbying_amount DESC
    LIMIT 100
''')
fec_spending = q('SELECT * FROM organization_fec_disbursements LIMIT 100')
print(f'Policy links (IRS org -> bills via LDA):   {conn.execute("SELECT COUNT(*) FROM organization_policy_links").fetchone()[0]:,} rows')
print(f'FEC spending (IRS org -> disbursements):   {conn.execute("SELECT COUNT(*) FROM organization_fec_disbursements").fetchone()[0]:,} rows')
print()
print('Top lobbying orgs by reported spend -> bill:')
policy_links.head(20)


Policy links (IRS org -> bills via LDA):   1,180,680 rows
FEC spending (IRS org -> disbursements):   0 rows

Top lobbying orgs by reported spend -> bill:


,ein,client_name,bill_id,title,filing_year,reported_lobbying_amount
0,361520690,NATIONAL ASSOCIATION OF REALTORS,118-hr-419,Revitalizing Downtowns Act,2024,32010000.0
1,361520690,NATIONAL ASSOCIATION OF REALTORS,118-hr-634,FAIRNESS in Flood Insurance Act of 2023,2024,32010000.0
2,361520690,NATIONAL ASSOCIATION OF REALTORS,118-hr-900,To amend the National Flood Insurance Act of 1...,2024,32010000.0
3,361520690,NATIONAL ASSOCIATION OF REALTORS,118-hr-1059,SECURE Notarization Act of 2023,2024,32010000.0
4,361520690,NATIONAL ASSOCIATION OF REALTORS,118-hr-1087,DEPOSIT Act,2024,32010000.0
5,361520690,NATIONAL ASSOCIATION OF REALTORS,118-hr-1165,Data Privacy Act of 2023,2024,32010000.0
6,361520690,NATIONAL ASSOCIATION OF REALTORS,118-hr-1307,To repeal the mandatory flood insurance covera...,2024,32010000.0
7,361520690,NATIONAL ASSOCIATION OF REALTORS,118-hr-1309,To require the use of replacement cost value i...,2024,32010000.0
8,361520690,NATIONAL ASSOCIATION OF REALTORS,118-hr-1312,Deerfield River Wild and Scenic River Study Ac...,2024,32010000.0
9,361520690,NATIONAL ASSOCIATION OF REALTORS,118-hr-1321,More Homes on the Market Act,2024,32010000.0


## Congress bills

Bills are in stub mode (title and bill type only). Call `congress.collect_bill_detail(bill_id)` for full detail on a specific bill.


In [9]:
# Bills are in stub mode (introduced_date, policy_area not populated from Congress API)
# Browse by type or search by bill number using lobbying_bill_facts for cross-reference
print(f'Total bills: {conn.execute("SELECT COUNT(*) FROM bills").fetchone()[0]:,}')
print(f'With policy_area: {conn.execute("SELECT COUNT(*) FROM bills WHERE policy_area IS NOT NULL").fetchone()[0]:,}')
print()
bills_by_type = q('''
    SELECT bill_type, COUNT(*) AS bills
    FROM bills
    GROUP BY bill_type
    ORDER BY bills DESC
''')
print('Bills by type:')
display(bills_by_type)

# Sample of actual bills
bills_sample = q('''
    SELECT bill_type, bill_number, title, introduced_date, latest_action
    FROM bills
    ORDER BY bill_number
    LIMIT 20
''')
bills_sample


Total bills: 16,565
With policy_area: 0

Bills by type:


,bill_type,bills
0,hr,10564
1,s,5649
2,hjres,230
3,sjres,122


,bill_type,bill_number,title,introduced_date,latest_action
0,hr,1,Lower Energy Costs Act,None,The Clerk was authorized to correct section nu...
1,s,1,Freedom to Vote Act,None,Read twice and referred to the Committee on Ru...
2,hjres,1,Proposing an amendment to the Constitution of ...,None,Referred to the House Committee on the Judiciary.
3,sjres,1,A joint resolution proposing amendments to the...,None,Read twice and referred to the Committee on th...
4,hr,2,Secure the Border Act of 2023,None,Referred to the Subcommittee on Social Security.
5,s,2,Ban Gambling on Elections Act of 2024,None,Read twice and referred to the Committee on Ag...
6,hjres,2,Proposing an amendment to the Constitution of ...,None,Referred to the House Committee on the Judiciary.
7,sjres,2,A joint resolution proposing an amendment to t...,None,Read twice and referred to the Committee on th...
8,hr,3,Reserved for the Speaker.,None,Introduced in House
9,s,3,Improving Access to Advance Care Planning Act,None,Read twice and referred to the Committee on Fi...
